In [23]:
import polars as pl
import bz2
import gzip as gz
import matplotlib.pyplot as plt
import numpy as np
import re
from collections import Counter

## Converting Data Types and formats

In [24]:
df = pl.scan_parquet("/home/ma/a/alb25/Project/thesis_code/data/raw/*")

In [25]:
# Adding the user type columns

def add_user_type(column):
    return (pl.when(pl.col(column).str.contains(r"^U\d+@")).then(pl.lit("human"))
        .when(pl.col(column).str.contains(r"^C\d+\$@")).then(pl.lit("machine"))
        .when(pl.col(column).str.contains(r"^(SYSTEM|LOCAL SERVICE|NETWORK SERVICE)@")).then(pl.lit("system"))
        .when(pl.col(column).str.contains(r"^ANONYMOUS LOGON@")).then(pl.lit("anon"))
        .otherwise(pl.lit("other")))

df = df.with_columns([add_user_type("source_user@domain").alias("source_user_type"),
    add_user_type("destination_user@domain").alias("destination_user_type")])

In [26]:
# converting the tiem column to a good format being a duration
# EDA revealed 58 days of data
df = df.with_columns(pl.col('time').cast(pl.Int64).alias('time'))
df = df.with_columns(pl.duration(seconds=pl.col('time')).alias('time'))

In [27]:
# Converting the dataframe success column to a boolean
# EDA revealed that it can only take 2 values Success and Failure
df = df.with_columns(pl.when(pl.col('success/failure') == 'Success').then(True).otherwise(False).alias('success')).drop('success/failure')

In [28]:
# Converting columns to categorical
for column_name in ('authentication_orientation', 'authentication_type', 'logon_type', 'source_user_type', 'destination_user_type'):
    col_vals = df.select(column_name).unique().collect().to_series().to_list()
    col_vals = pl.Enum(col_vals)
    df = df.with_columns(pl.col(column_name).cast(col_vals).alias(column_name))

### Creating a train test validaiton split

2 week for test 1 week validation the rest for train

In [33]:
df = df.with_columns(pl.col('time').dt.total_days().alias('day'))

In [54]:
test_df = df.filter(pl.col('day') >= 44)
validation_df = df.filter((pl.col('day') >= 37) & (pl.col('day') < 44))
train_df = df.filter(pl.col('day') < 37)

In [60]:
for df_name, d in {'test_df' : test_df, 'train_df' : train_df, 'validation_df' :validation_df}.items():
    d = d.drop('day')
    d = d.sort(['source_user@domain', 'time'])
    d.sink_parquet(f'/home/ma/a/alb25/Project/thesis_code/data/processed/{df_name}.parquet',
                   compression="lz4", statistics=True, row_group_size=250_000)
